In [2]:
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent
from openai import OpenAI

load_dotenv()

# ---- LangSmith tracing ----
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY is missing. Add it to .env, then rerun this cell.")

os.environ["LANGSMITH_API_KEY"] = langsmith_api_key
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "physio")
if os.getenv("LANGSMITH_ENDPOINT"):
    os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
if os.getenv("LANGSMITH_WORKSPACE_ID"):
    os.environ["LANGSMITH_WORKSPACE_ID"] = os.getenv("LANGSMITH_WORKSPACE_ID")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def get_available_session_slots() -> str:
    """Return available clinic session slots (static response)."""
    return (
        "Available slots at V33 (El Waha, Cairo):\n"
        "- Sunday: 10:00 AM, 12:00 PM, 5:00 PM\n"
        "- Monday: 11:00 AM, 2:00 PM, 6:00 PM\n"
        "- Tuesday: 9:30 AM, 1:30 PM, 4:30 PM\n"
        "- Wednesday: 10:30 AM, 3:00 PM\n"
        "- Thursday: 12:30 PM, 5:30 PM"
    )

def search_internet(query: str, max_results: int = 5) -> str:
    """Search internet for physiotherapy/injury information."""
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        tools=[{"type": "web_search_preview"}],
        tool_choice={"type": "web_search_preview"},
        input=f"Return up to {max_results} concise reliable findings for: {query}",
    )
    return response.output_text

def search_youtube(workout_query: str, max_results: int = 5) -> str:
    """Search YouTube videos for exercises/workouts using web search."""
    yt_query = f"site:youtube.com physiotherapy exercise video {workout_query}"
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        tools=[{"type": "web_search_preview"}],
        tool_choice={"type": "web_search_preview"},
        input=f"Find up to {max_results} useful YouTube videos for: {yt_query}. Return title + link.",
    )
    return response.output_text

def get_user_details(user_name: str) -> str:
    """Return user profile details (static response for user 'ziad')."""
    if user_name.strip().lower() != "ziad":
        return "No stored profile found for this user yet (static demo supports only user 'ziad')."

    return (
        "User: Ziad\n"
        "Diagnosis: Patellofemoral pain syndrome (right knee)\n"
        "Medical history: mild recurrent low-back pain, no major surgeries\n"
        "Remaining sessions: 4 out of 12\n"
        "Current plan: quad strengthening, hip stability, mobility drills"
    )

def schedule_appointment(user_name: str, preferred_day: str, preferred_time: str) -> str:
    """Schedule appointment confirmation (static response)."""
    return (
        f"Appointment confirmed for {user_name} at V33 (El Waha, Cairo) on {preferred_day} at {preferred_time}. "
        "Please arrive 10 minutes early."
    )

system_prompt = """You are a general physiotherapy assistant for V33 clinic in El Waha, Cairo.
You help users with pain/injury/physiotherapy questions and clinic operations.

Clinic context:
- Name: V33
- Location: El Waha, Cairo
- Focus: physiotherapy, rehabilitation, and injury recovery

,Tool usage policy:
1) If user asks about sessions/availability -> use get_available_session_slots.
2) If user asks clinical education or injury/pain info -> use search_internet.
3) If user asks for exercise/workout videos -> use search_youtube.
4) If user asks about his profile/history/sessions and provides a name -> use get_user_details.
5) If user asks to book/confirm appointment -> use schedule_appointment.

Safety policy:
- Give educational guidance only; do not provide definitive diagnosis.
- Escalate urgent red flags to immediate in-person care.
"""

agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    system_prompt=system_prompt,
    tools=[
        get_available_session_slots,
        search_internet,
        search_youtube,
        get_user_details,
        schedule_appointment,
    ],
)


In [3]:

# Demo queries
q1 = agent.invoke(
    {"messages": [{"role": "user", "content": "I have shoulder pain after gym presses, what should I do first?"}]},
    config={"run_name": "physio-general-clinical", "tags": ["physio", "general-agent"]},
)
print("Clinical answer:\n", q1["messages"][-1].content)


Clinical answer:
 [{'type': 'text', 'text': 'For shoulder pain after gym presses, follow these steps:\n\n1. **Rest**: Avoid any movements or exercises that aggravate the pain.\n2. **Ice**: Apply ice to the affected area for 15-20 minutes every few hours to reduce inflammation.\n3. **Compression**: Use a compression bandage if necessary to support the shoulder.\n4. **Elevation**: Keep your shoulder elevated to help reduce swelling.\n5. **Gentle Range of Motion**: If pain allows, gently move the shoulder through its range of motion without resistance.\n\nIf the pain persists or worsens, consider consulting a healthcare professional. Would you like more detailed information on shoulder injuries?', 'annotations': [], 'id': 'msg_02302e71518604110069a1994b119481a1876f232b7ffb3ac8'}]


In [4]:

q2 = agent.invoke(
    {"messages": [{"role": "user", "content": "Show me youtube videos for knee mobility drills"}]},
    config={"run_name": "physio-general-youtube", "tags": ["physio", "general-agent", "youtube"]},
)
print("\nYouTube answer:\n", q2["messages"][-1].content)



YouTube answer:
 [{'type': 'text', 'text': 'Here are five useful YouTube videos demonstrating physiotherapy exercises for knee mobility drills:\n\n1. **AP Joint Mobilisation for Knee Flexion | Tim Keeley | Physio REHAB**\n   - This video showcases anterior-posterior joint mobilisations to enhance knee flexion range of motion.\n   - [Watch here](https://www.youtube.com/watch?v=uqhqt2D1eR4)\n\n2. **Passive Knee Mobilisation MWM Physiotherapy Techniques**\n   - Demonstrates passive knee mobilisation with mobilisations with movement (MWM) techniques to improve knee mobility.\n   - [Watch here](https://www.youtube.com/watch?v=hBOicxjVAgA)\n\n3. **Knee Mobility Drills for Flexibility and Strength**\n   - A comprehensive guide to exercises aimed at increasing knee flexibility and strength.\n   - [Watch here](https://www.youtube.com/watch?v=E_VyLQHngAI)\n\n4. **Knee Rehabilitation Exercises for Improved Mobility**\n   - Focuses on rehabilitation exercises designed to enhance knee mobility pos

In [5]:

q3 = agent.invoke(
    {"messages": [{"role": "user", "content": "I am ziad, how many sessions do I have left and book me Thursday at 5:30 PM"}]},
    config={"run_name": "physio-general-profile-booking", "tags": ["physio", "general-agent", "booking"]},
)
print("\nProfile/booking answer:\n", q3["messages"][-1].content)


Profile/booking answer:
 [{'type': 'text', 'text': 'You have 4 sessions left out of 12. Your appointment is confirmed for Thursday at 5:30 PM at V33, El Waha, Cairo. Please arrive 10 minutes early.', 'annotations': [], 'id': 'msg_0bc5ab7483fb99fa0069a199d452888191971991bbc823a732'}]


In [10]:
q3

{'messages': [HumanMessage(content='I am ziad, how many sessions do I have left and book me Thursday at 5:30 PM', additional_kwargs={}, response_metadata={}, id='be54f35a-6348-4130-accb-12d01835f99a'),
  AIMessage(content=[{'arguments': '{"user_name":"ziad"}', 'call_id': 'call_iZhxy1ga4ZxR6ESGJA8hHI7n', 'name': 'get_user_details', 'type': 'function_call', 'id': 'fc_0bc5ab7483fb99fa0069a199d3074c8191ba4ac66d3dea9838', 'status': 'completed'}, {'arguments': '{"user_name":"ziad","preferred_day":"Thursday","preferred_time":"5:30 PM"}', 'call_id': 'call_eFU0G57zabdHCjVHLUUWAJ4e', 'name': 'schedule_appointment', 'type': 'function_call', 'id': 'fc_0bc5ab7483fb99fa0069a199d307608191af3489b0fbc04710', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0bc5ab7483fb99fa0069a199d1152481919155884e9614fcb1', 'created_at': 1772198353.0, 'metadata': {}, 'model': 'gpt-4o-mini-2024-07-18', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider':